# Stage 5: CNN & ViT Baselines (B1, B2, B4)
**Environment:** Kaggle Notebook (GPU T4)

In [ ]:
import numpy as np, pandas as pd, cv2, os, glob, json, time
from typing import Dict, List
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T, torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, roc_curve
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.notebook import tqdm
from PIL import Image

%matplotlib inline
plt.rcParams['figure.dpi'] = 120; sns.set_style('whitegrid')

INPUT_DIR = '/kaggle/input/artifact-dataset'
MODEL_DIR = '/kaggle/working/models'
os.makedirs(MODEL_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

CONFIG = {'batch_size': 32, 'epochs': 30, 'lr': 1e-4, 'wd': 1e-4,
          'patience': 8, 'n_per_gen': 200, 'random_state': 42}

In [ ]:
# Cell 2: Dataset Classes & Data Loading
meta_files = glob.glob(os.path.join(INPUT_DIR, '**', 'metadata.csv'), recursive=True)
all_dfs = []
for mf in meta_files:
    df = pd.read_csv(mf)
    gen_dir = os.path.dirname(mf)
    df['generator'] = os.path.basename(gen_dir)
    df['image_path'] = df['image_path'].apply(lambda p: os.path.join(gen_dir, p) if not os.path.isabs(p) else p)
    all_dfs.append(df)
if not all_dfs:
    image_exts = {'.png','.jpg','.jpeg'}
    records = []
    for root,_,files in os.walk(INPUT_DIR):
        for f in files:
            if os.path.splitext(f)[1].lower() in image_exts:
                full = os.path.join(root,f)
                parts = os.path.relpath(root,INPUT_DIR).lower().split(os.sep)
                is_real = 'real' in parts
                records.append({'image_path':full,'target':0 if is_real else 1,'generator':parts[-1]})
    metadata_df = pd.DataFrame(records)
else:
    metadata_df = pd.concat(all_dfs, ignore_index=True)

# Sample per generator
sampled = [g_df.sample(n=min(len(g_df), CONFIG['n_per_gen']), random_state=42)
           for _, g_df in metadata_df.groupby('generator')]
metadata_df = pd.concat(sampled, ignore_index=True)
metadata_df = metadata_df[metadata_df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(f'Images: {len(metadata_df)} (Real:{(metadata_df["target"]==0).sum()}, Fake:{(metadata_df["target"]==1).sum()})')

train_val_df, test_df = train_test_split(metadata_df, test_size=0.2, stratify=metadata_df['target'], random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.15, stratify=train_val_df['target'], random_state=42)
print(f'Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}')

class RGBDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'], cv2.IMREAD_COLOR)
        if img is None: img = np.zeros((224,224,3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224,224), interpolation=cv2.INTER_CUBIC)
        img = Image.fromarray(img)
        if self.transform: img = self.transform(img)
        else: img = T.ToTensor()(img)
        return img, int(row['target'])

class MagnitudeDataset(Dataset):
    _LR,_LG,_LB = 0.2989,0.5870,0.1140
    def __init__(self, df, size=224):
        self.df = df.reset_index(drop=True); self.size = size
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'], cv2.IMREAD_COLOR)
        if img is None: return torch.zeros(3,self.size,self.size), int(row['target'])
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float64)
        gray = self._LR*rgb[:,:,0]+self._LG*rgb[:,:,1]+self._LB*rgb[:,:,2]
        gray = cv2.resize(gray, (self.size,self.size), interpolation=cv2.INTER_CUBIC)
        fft = np.fft.fftshift(np.fft.fft2(gray))
        mag = np.log1p(np.abs(fft)); mag = (mag-mag.min())/(mag.max()-mag.min()+1e-8)
        return torch.FloatTensor(np.stack([mag]*3).astype(np.float32)), int(row['target'])

train_tf = T.Compose([T.RandomHorizontalFlip(), T.RandomRotation(10),
    T.ColorJitter(0.1,0.1), T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
val_tf = T.Compose([T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
print('Datasets ready.')

In [ ]:
# Cell 3: Trainer & Model Builders
def build_resnet50():
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 1))
    return m

def build_vit():
    m = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
    m.heads = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.heads[0].in_features, 1))
    return m

def train_eval(model, train_l, val_l, test_l, name, cfg):
    model = model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    opt = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['epochs'])
    best_auc, pat = 0, 0
    for epoch in tqdm(range(cfg['epochs']), desc=name):
        model.train()
        for X,y in train_l:
            X,y = X.to(DEVICE), y.float().to(DEVICE)
            opt.zero_grad(); loss = criterion(model(X).squeeze(-1), y); loss.backward(); opt.step()
        sched.step()
        model.eval(); vp,vt = [],[]
        with torch.no_grad():
            for X,y in val_l:
                vp.append(torch.sigmoid(model(X.to(DEVICE)).squeeze(-1)).cpu().numpy()); vt.append(y.numpy())
        vp,vt = np.concatenate(vp), np.concatenate(vt)
        vauc = roc_auc_score(vt,vp) if len(np.unique(vt))>1 else 0
        if vauc > best_auc:
            best_auc = vauc; pat = 0
            torch.save(model.state_dict(), os.path.join(MODEL_DIR, f'{name}_best.pth'))
        else:
            pat += 1
            if pat >= cfg['patience']: print(f'Early stop {name} ep {epoch+1}'); break
    model.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'{name}_best.pth'), weights_only=True))
    model.eval(); tp,tt = [],[]
    with torch.no_grad():
        for X,y in test_l:
            tp.append(torch.sigmoid(model(X.to(DEVICE)).squeeze(-1)).cpu().numpy()); tt.append(y.numpy())
    tp,tt = np.concatenate(tp), np.concatenate(tt)
    acc = accuracy_score(tt,(tp>0.5).astype(int)); auc = roc_auc_score(tt,tp)
    n_p = sum(p.numel() for p in model.parameters())
    print(f'{name}: Acc={acc:.4f} AUC={auc:.4f} Params={n_p:,}')
    print(classification_report(tt,(tp>0.5).astype(int), target_names=['Real','Fake']))
    return {'model':name,'test_accuracy':float(acc),'test_auc':float(auc),'n_parameters':n_p,'preds':tp,'targets':tt}

print('Trainer ready.')

In [ ]:
# Cell 4: Train All Baselines
nw = 2
rgb_train = DataLoader(RGBDataset(train_df, train_tf), batch_size=32, shuffle=True, num_workers=nw)
rgb_val = DataLoader(RGBDataset(val_df, val_tf), batch_size=32, num_workers=nw)
rgb_test = DataLoader(RGBDataset(test_df, val_tf), batch_size=32, num_workers=nw)
mag_train = DataLoader(MagnitudeDataset(train_df), batch_size=32, shuffle=True, num_workers=nw)
mag_val = DataLoader(MagnitudeDataset(val_df), batch_size=32, num_workers=nw)
mag_test = DataLoader(MagnitudeDataset(test_df), batch_size=32, num_workers=nw)

all_results = {}
print('='*50 + '\nB1: ResNet-50 RGB\n' + '='*50)
all_results['B1'] = train_eval(build_resnet50(), rgb_train, rgb_val, rgb_test, 'ResNet-50-RGB (B1)', CONFIG)
print('='*50 + '\nB2: ResNet-50 Magnitude\n' + '='*50)
all_results['B2'] = train_eval(build_resnet50(), mag_train, mag_val, mag_test, 'ResNet-50-Mag (B2)', CONFIG)
print('='*50 + '\nB4: ViT-B/16 RGB\n' + '='*50)
all_results['B4'] = train_eval(build_vit(), rgb_train, rgb_val, rgb_test, 'ViT-B/16-RGB (B4)', CONFIG)

In [ ]:
# Cell 5: Compare
for fname, key in [('results_kan.json','KAN-Phase'),('results_b3.json','B3')]:
    p = os.path.join(MODEL_DIR, fname)
    if os.path.exists(p):
        with open(p) as f: d = json.load(f)
        all_results[key] = {'model':d['model'],'test_accuracy':d['test_accuracy'],'test_auc':d['test_auc'],'n_parameters':d['n_parameters']}

rows = [{'Model':r['model'],'AUC':r['test_auc'],'Accuracy':r['test_accuracy'],'Parameters':r['n_parameters']}
        for r in all_results.values()]
comp_df = pd.DataFrame(rows).sort_values('AUC', ascending=False)
print('\nFULL COMPARISON:'); print(comp_df.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Baseline Comparison', fontweight='bold')
colors = ['#e74c3c','#3498db','#2ecc71','#9b59b6','#f39c12']
ax1.barh(comp_df['Model'], comp_df['AUC'], color=colors[:len(comp_df)])
ax1.set_xlabel('AUC'); ax1.set_xlim(0,1.05)
for bar,auc in zip(ax1.patches, comp_df['AUC']):
    ax1.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2, f'{auc:.3f}', va='center', fontsize=8)
ax2.scatter(comp_df['Parameters'], comp_df['AUC'], s=100, c=colors[:len(comp_df)], zorder=5)
for _,row in comp_df.iterrows():
    ax2.annotate(row['Model'][:15], (row['Parameters'],row['AUC']), fontsize=7, xytext=(5,5), textcoords='offset points')
ax2.set_xlabel('Parameters'); ax2.set_ylabel('AUC'); ax2.set_xscale('log')
plt.tight_layout(); plt.show()

# ROC
fig, ax = plt.subplots(figsize=(8,6))
for key,c in [('B1','#3498db'),('B2','#e67e22'),('B4','#9b59b6')]:
    if key in all_results and 'preds' in all_results[key]:
        r = all_results[key]; fpr,tpr,_ = roc_curve(r['targets'],r['preds'])
        ax.plot(fpr,tpr,label=f"{r['model']} ({r['test_auc']:.3f})",lw=2,color=c)
ax.plot([0,1],[0,1],'k--',alpha=0.3); ax.set_title('ROC Curves'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

save = {k:{kk:v for kk,v in r.items() if kk not in ('preds','targets')} for k,r in all_results.items()}
with open(os.path.join(MODEL_DIR,'all_results.json'),'w') as f: json.dump(save,f,indent=2)
comp_df.to_csv(os.path.join(MODEL_DIR,'all_baselines_comparison.csv'), index=False)
print('All results saved.')